In [8]:
import os
import duckdb
import pandas as pd
from pathlib import Path

# Resolve project root regardless of where the notebook is opened from
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR = PROJECT_ROOT / "sql"

# Change to project root so relative paths like 'data/...' work in SQL
os.chdir(PROJECT_ROOT)

print(f"DuckDB version: {duckdb.__version__}")
print(f"Project root:   {PROJECT_ROOT}")
print(f"CWD now:        {os.getcwd()}")

DuckDB version: 1.5.2
Project root:   /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling
CWD now:        /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling


In [9]:
con = duckdb.connect(str(DATA_DIR / "criteo.duckdb"))
print("Connected.")

Connected.


In [7]:
import os
print("CWD:", os.getcwd())
print("CSV exists at absolute path:", (DATA_DIR / "criteo-uplift.csv").exists())

CWD: /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/notebooks
CSV exists at absolute path: True


In [10]:
%%time
con.execute("""
    CREATE OR REPLACE TABLE criteo AS
    SELECT * FROM read_csv_auto('data/criteo-uplift.csv')
""")
print("Load complete.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Load complete.
CPU times: user 15.2 s, sys: 1.68 s, total: 16.9 s
Wall time: 2.39 s


In [11]:
n_rows = con.execute("SELECT COUNT(*) FROM criteo").fetchone()[0]
print(f"Rows: {n_rows:,}")
assert n_rows == 13_979_592, f"Expected 13,979,592 rows, got {n_rows:,}"
print("Row count matches Criteo spec.")

Rows: 13,979,592
Row count matches Criteo spec.


In [12]:
schema = con.execute("DESCRIBE criteo").fetchdf()
print(schema)

# Save to sql/schema.txt
schema.to_csv(SQL_DIR / "schema.txt", sep="\t", index=False)
print(f"Schema saved to {SQL_DIR / 'schema.txt'}")

   column_name column_type null   key default extra
0           f0      DOUBLE  YES  None    None  None
1           f1      DOUBLE  YES  None    None  None
2           f2      DOUBLE  YES  None    None  None
3           f3      DOUBLE  YES  None    None  None
4           f4      DOUBLE  YES  None    None  None
5           f5      DOUBLE  YES  None    None  None
6           f6      DOUBLE  YES  None    None  None
7           f7      DOUBLE  YES  None    None  None
8           f8      DOUBLE  YES  None    None  None
9           f9      DOUBLE  YES  None    None  None
10         f10      DOUBLE  YES  None    None  None
11         f11      DOUBLE  YES  None    None  None
12   treatment      BIGINT  YES  None    None  None
13  conversion      BIGINT  YES  None    None  None
14       visit      BIGINT  YES  None    None  None
15    exposure      BIGINT  YES  None    None  None
Schema saved to /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/sql/schema.txt

In [13]:
split = con.execute("""
    SELECT
        treatment,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (), 4) AS pct
    FROM criteo
    GROUP BY treatment
    ORDER BY treatment
""").fetchdf()
print(split)

   treatment         n   pct
0          0   2096937  0.15
1          1  11882655  0.85


In [14]:
outcomes = con.execute("""
    SELECT
        treatment,
        AVG(visit)        AS visit_rate,
        AVG(conversion)   AS conversion_rate,
        AVG(exposure)     AS exposure_rate,
        COUNT(*)          AS n
    FROM criteo
    GROUP BY treatment
    ORDER BY treatment
""").fetchdf()
print(outcomes)

   treatment  visit_rate  conversion_rate  exposure_rate         n
0          0    0.038201         0.001938       0.000000   2096937
1          1    0.048543         0.003089       0.036037  11882655


In [15]:
visit_lift = outcomes.loc[outcomes.treatment==1, "visit_rate"].iloc[0] - outcomes.loc[outcomes.treatment==0, "visit_rate"].iloc[0]
conv_lift  = outcomes.loc[outcomes.treatment==1, "conversion_rate"].iloc[0] - outcomes.loc[outcomes.treatment==0, "conversion_rate"].iloc[0]
print(f"Raw visit lift:      {visit_lift:.6f}")
print(f"Raw conversion lift: {conv_lift:.6f}")

con.close()
print("Connection closed.")

Raw visit lift:      0.010342
Raw conversion lift: 0.001152
Connection closed.
